In [2]:
import importlib
from pathlib import Path
from torch.utils.data import DataLoader
import h5py
import time
import numpy as np

In [3]:
import sys
from pathlib import Path

project_root = Path("..").resolve()  # points to /project
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from functions_library import *

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model_library_path = "C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/"
z_mass_hist_path = "C:/Users/jonas/Coding/Master/Main/csv_files/"
z_mass_files = ["Z_mass_EGAM1.csv", "Z_mass_EGAM7.csv"]
full_z_mass_hist_path = [z_mass_hist_path + file for file in z_mass_files]
base_string = "C:/Users/jonas/Coding/Master/Main/h5/"
data_strings = ["MC_EGAM1", "MC_EGAM7", "DATA_EGAM"]
name_strings = data_strings + ["MC_EGAM"]
full_strings = [base_string + data_string for data_string in data_strings]
h5_files_dir = data_loading.create_path_lists(full_strings, outputs=name_strings, mixing_list = [(0, 1)], seed=0)
h5_files_egam1 = h5_files_dir[name_strings[0]]
h5_files_egam7 = h5_files_dir[name_strings[1]]
h5_files_egam = h5_files_dir[name_strings[-1]]
h5_files_data = h5_files_dir[name_strings[-2]]

In [5]:
dataset_target = dataset_classes.ZPairDataset(
    h5_files_egam1,
    valid_fields = {"egammas": ["truthOrigin"]},
    y_source = "truth",
    y_field = ["pt", "eta", "phi", "e"],
)
loader = DataLoader(misc_functions._TargetOnlyDataset(dataset_target), batch_size=4096*16, num_workers=8, pin_memory=False)
targets = []
for batch in loader:
    print(batch.shape)
    print(batch)
    break

torch.Size([65536, 2, 4])
tensor([[[ 2.5369e+04,  1.9180e-01,  1.4868e+00,  2.5837e+04],
         [ 3.5712e+04,  1.0922e+00, -1.1542e+00,  5.9217e+04]],

        [[ 1.2658e+04,  2.0953e-01,  1.4015e+00,  1.2937e+04],
         [ 9.7665e+04,  1.2270e+00, -1.1640e+00,  1.8089e+05]],

        [[ 3.0605e+04,  2.0329e+00, -6.2664e-01,  1.1886e+05],
         [        nan,         nan,         nan,         nan]],

        ...,

        [[ 8.4188e+04,  1.2752e+00,  7.6509e-01,  1.6243e+05],
         [ 3.7224e+04,  1.1270e+00,  2.6503e+00,  6.3472e+04]],

        [[ 8.4188e+04,  1.2752e+00,  7.6509e-01,  1.6243e+05],
         [ 5.2641e+03,  1.1628e+00, -2.0360e+00,  9.2431e+03]],

        [[ 8.4188e+04,  1.2752e+00,  7.6509e-01,  1.6243e+05],
         [ 1.2140e+04,  1.1883e+00, -2.0579e+00,  2.1768e+04]]])


In [9]:
model_particle_size = []
model_names = ["origin", "pt", "eta", "phi", "e"]
for name in model_names:
    if name == "origin":
        small_dataset = dataset_classes.ZPairDataset(
            h5_files_egam[0],
            exclude_features = ["truth"],
            exclude_fields = ["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
        )
    else:
        small_dataset = dataset_classes.ZPairDataset(
            h5_files_egam[0],
            exclude_features = ["truth"],
            exclude_fields = ["truthOrigin", "truthType", name],
        )
    particle_fields = [field for field in small_dataset.fields if not field.startswith("eventwise_")]
    event_fields = [field for field in small_dataset.fields if field.startswith("eventwise_")]
    model_particle_size.append(len(particle_fields) // 2)
    if len(particle_fields) % 2 != 0:
        raise ValueError("Pair particle fields are not evenly split between the two particles.")


In [10]:
bins = np.arange(55.05, 135.05, 0.1)
model_paths = [model_library_path + "model_" + name + "_pair.pth" for name in model_names]
models = {}
for name, path, size in zip(model_names, model_paths, model_particle_size):
    print(f"Loading model {name} from {path}")
    if name == "origin":
        output_size = 1
    else:
        output_size = 4
    model = neural_models.ParticlePairNetwork(
        size,
        len(event_fields),
        particle_hidden_sizes=[1024, 256, 256, 256, 256, 256, 64, 8],
        event_hidden_sizes=[64, 64, 16, 4],
        consolidation_hidden_sizes=[32, 32, 32, 8],
        output_size=output_size,
    ).to(device)
    model.load_state_dict(torch.load(path))
    model.eval()
    models[name] = model


Loading model origin from C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/model_origin_pair.pth
Loading model pt from C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/model_pt_pair.pth
Loading model eta from C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/model_eta_pair.pth
Loading model phi from C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/model_phi_pair.pth
Loading model e from C:/Users/jonas/Coding/Master/Main/Masters_project/models_library/model_e_pair.pth


In [14]:
dataset_mean = dataset_classes.H5EgammaDataset(
    h5_files_egam[:3],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
)
means, stds = misc_functions.compute_mean_std(dataset_mean, batch_size=1024, num_workers=4, sample_size=10000)
dataset_mean.close()

dataset_origin = dataset_classes.ZPairDataset(
    h5_files_egam[:3],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
    y_source="truth",
    y_field=["pt"]
)
origin_loader = DataLoader(dataset_origin, batch_size=1024, shuffle=False, num_workers=4, prefetch_factor=2, persistent_workers=True)

time_start_o = time.perf_counter()
prediction_o = model_training.predict(models["origin"], origin_loader, device=device, means=means, stds=stds)
time_o = time.perf_counter() - time_start_o
dataset_origin.close()

dataset_mean = dataset_classes.H5EgammaDataset(
    h5_files_egam[:3],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "pt"],
)
means, stds = misc_functions.compute_mean_std(dataset_mean, batch_size=1024, num_workers=4, sample_size=10000)
dataset_mean.close()

dataset_pt = dataset_classes.ZPairDataset(
    h5_files_egam[:3],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "pt"],
    y_source="truth",
    y_field=["pt"]
)
pt_loader = DataLoader(dataset_pt, batch_size=1024, shuffle=False, num_workers=4, prefetch_factor=2, persistent_workers=True)

time_start_p = time.perf_counter()
prediction_p = model_training.predict(models["pt"], pt_loader, device=device, log_target=True, means=means, stds=stds)
time_p = time.perf_counter() - time_start_p
dataset_pt.close()

print(f"Time taken to validate model: {time_o:.2f} seconds")
print(f"Time taken to predict model: {time_p:.2f} seconds")

Prediction completed                                                                                                                                                                                                                                                                                        
Prediction completed                                                                                                                                                                                                                                                                                        
Time taken to validate model: 66.52 seconds
Time taken to predict model: 66.03 seconds


In [15]:
len(prediction_o), len(prediction_p)

(479903, 479903)

In [ ]:
test_dataset = dataset_classes.H5EgammaDataset(
    h5_files_egam,
    mix_files = h5_files_data,
    mixture_ratio = 0.8,
    exclude_features = ["truth"],
    exclude_fields = ["truthOrigin", "truthType"],
    fallback_target = ["passid_Medium"],
    y_source = "egammas",
    y_field = "truthOrigin",
)
sampler = data_loading.ShuffledContiguousBatchSampler(len(test_dataset), batch_size=1024, dataset=test_dataset)
test_loader = DataLoader(test_dataset, batch_sampler=sampler, num_workers=8, prefetch_factor=2, persistent_workers=True)
for batch_idx, (_) in enumerate(test_loader):
    if batch_idx == 5:
        start = time.perf_counter()
    if batch_idx == 25:
        end = time.perf_counter()
        break
print(f"Time taken to load 10 batches: {end - start:.4f} seconds")